# 11 · Table Statistics

**Layer:** cross-cutting &nbsp;|&nbsp; **Stage:** profiling &nbsp;|&nbsp; **Target:** `banking_gold.table_stats`

A **parameterised** notebook — the body of the pipeline's `for_each` task.
The pipeline iterates the `gold_tables` parameter and invokes this notebook
once per table with `TARGET_TABLE` bound to the current item, two at a time.

Profiles whatever table it is handed: row count, column count, null density
per column, and distinct-value cardinality for the low-cardinality columns.
Results are appended, so the table becomes a profile history across runs.


In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("medallion-table-stats").enableHiveSupport().getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "16")

RUN_DATE = "2026-09-03"
BATCH_ID = f"batch_{RUN_DATE.replace('-', '')}"
GOLD_DB  = "banking_gold"

# Bound by the pipeline's for_each task to each item of `params.gold_tables`.
# The literal below is the interactive-development default.
TARGET_TABLE = "banking_gold.account_balance_daily"

print(f"profiling {TARGET_TABLE}")

In [ ]:
df = spark.table(TARGET_TABLE)
row_count = df.count()
columns   = [c for c in df.columns if not c.startswith("_")]

print(f"rows    : {row_count}")
print(f"columns : {len(df.columns)} ({len(columns)} business, {len(df.columns) - len(columns)} lineage)")
df.printSchema()

In [ ]:
# Null density for every business column, in a single pass.
null_counts = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in columns
]).collect()[0].asDict()

profile = []
for col_name in columns:
    nulls = null_counts[col_name] or 0
    null_pct = round(100.0 * nulls / row_count, 2) if row_count else 0.0
    profile.append((col_name, str(df.schema[col_name].dataType.simpleString()), nulls, null_pct))

for name, dtype, nulls, pct in profile:
    bar = "#" * int(pct / 5)
    print(f"  {name:<28} {dtype:<12} nulls={nulls:<5} {pct:>6.2f}%  {bar}")

In [ ]:
# Cardinality, restricted to columns cheap enough to count exactly.
LOW_CARD_TYPES = ("string", "boolean", "date")

cardinality = {}
for col_name in columns:
    if df.schema[col_name].dataType.simpleString() in LOW_CARD_TYPES:
        cardinality[col_name] = df.select(col_name).distinct().count()

for name, n in sorted(cardinality.items(), key=lambda kv: -kv[1]):
    print(f"  {name:<28} distinct={n}")

In [ ]:
stats_rows = [
    (
        BATCH_ID, RUN_DATE, TARGET_TABLE,
        col_name, dtype, int(row_count), int(nulls), float(null_pct),
        int(cardinality.get(col_name, -1)),
    )
    for col_name, dtype, nulls, null_pct in profile
]

stats = spark.createDataFrame(
    stats_rows,
    "batch_id string, run_date string, table_name string, column_name string, "
    "data_type string, table_row_count bigint, null_count bigint, "
    "null_pct double, distinct_count bigint",
).withColumn("profiled_at", F.current_timestamp())

(
    stats.write.mode("append").format("parquet")
    .saveAsTable(f"{GOLD_DB}.table_stats")
)

print(f"appended {stats.count()} column profiles for {TARGET_TABLE}")

In [ ]:
spark.table(f"{GOLD_DB}.table_stats") \
     .filter((F.col("batch_id") == BATCH_ID) & (F.col("table_name") == TARGET_TABLE)) \
     .select("column_name", "data_type", "null_count", "null_pct", "distinct_count") \
     .orderBy(F.desc("null_pct")) \
     .show(50, truncate=False)

print("=" * 60)
print(f"PROFILED table={TARGET_TABLE} rows={row_count} columns={len(columns)}")
print("=" * 60)